In [5]:
import os
import re
import json
import time
import joblib
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from IPython.display import display

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import mean_squared_error, accuracy_score, precision_score, make_scorer
from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import (
    RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor,
    RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier,
)
from sklearn.pipeline import Pipeline

load_dotenv()
USDA_API_KEY = os.getenv("USDA_API_KEY")
DATA = "../datasets"
OUT = "."
CACHE_PATH = "usda_raw_cache.json"


# Исследование и прогнозирование качества блюд (Food & Nutrition)

## 1. Подготовка и фильтрация данных (Feature Selection)

**Физика задачи:** Исходная таблица  содержит 680 признаков, среди которых перемешаны ингредиенты, географические теги (, ), кулинарные категории (, ), макронутриенты (, ) и способы термической обработки (, , ).

Для обучения модели, способной оценивать блюдо исключительно по продуктовой корзине пользователя, необходимо отфильтровать все «не-ингредиентные» колонки. Для этого мы используем поваренную книгу , где в поле  авторы зафиксировали только реальные продукты.

### 1.1. Загрузка исходных данных

Загружаем структурированный JSON-архив и плоскую таблицу признаков.

In [6]:
with open("../datasets/full_format_recipes.json", "r", encoding="utf-8") as f:
    recipes_data = json.load(f)

df_raw = pd.read_csv("../datasets/epi_r.csv")

print(f"Всего рецептов в JSON: {len(recipes_data)}")
print(f"Размер исходной таблицы: {df_raw.shape} (строк, колонок)")


Всего рецептов в JSON: 20130
Размер исходной таблицы: (20052, 680) (строк, колонок)


### 1.2. Формирование кулинарного словаря (White-list ингредиентов)

Извлекаем все слова из списков ингредиентов рецептов в JSON, формируя словарь слов, которые реально используются в кулинарии.

In [7]:
ingredient_words = set()
for recipe in recipes_data:
    ingredients = recipe.get("ingredients")
    if ingredients:
        for item in ingredients:
            words = re.findall(r"\b[a-z]+\b", item.lower())
            ingredient_words.update(words)

print(f"Уникальных кулинарных слов в базе рецептов: {len(ingredient_words)}")


Уникальных кулинарных слов в базе рецептов: 6157


### 1.3. Фильтрация признаков и приведение имен колонок к стандарту пользовательского ввода

1. Приводим составные имена ( -> ,  -> ) к стандартам пользовательского ввода и технического задания.
2. Исключаем из обучающей выборки черный список тегов: географию, макронутриенты, категории и способы приготовления.
3. Оставляем только те колонки, чьи названия полностью состоят из реальных кулинарных слов.

In [8]:
# 1. Алиасы к стандарту пользовательского ввода
df = df_raw.rename(columns={"milk/cream": "milk", "jam or jelly": "jam"}).copy()

# 2. Черный список (не-ингредиентные признаки)
blacklist = {
    "title", "calories", "protein", "fat", "sodium",
    "leftovers", "california", "dominican republic",
    "low cholesterol", "pan-fry", "fruit", "stew", "braise"
}

# 3. Фильтрация колонок
clean_cols = []
for col in df.columns:
    if col == "rating":
        clean_cols.append(col)
        continue
    if col in blacklist:
        continue
    words = re.findall(r"[a-z]+", col.lower())
    if words and all(w in ingredient_words for w in words):
        clean_cols.append(col)

df_clean = df[clean_cols].copy()

print(f"Исходных колонок: {df.shape[1]}")
print(f"Чистых колонок (ингредиенты + rating): {df_clean.shape[1]}")

# 4. Контрольная валидация состава признаков
assert "title" not in df_clean.columns
assert "protein" not in df_clean.columns
assert "fruit" not in df_clean.columns
assert "leftovers" not in df_clean.columns
assert "california" not in df_clean.columns
assert "pan-fry" not in df_clean.columns
assert "milk" in df_clean.columns
assert "jam" in df_clean.columns
print(" Фильтрация признаков успешно завершена!")


Исходных колонок: 680
Чистых колонок (ингредиенты + rating): 498
 Фильтрация признаков успешно завершена!


### 1.4. Обработка пропусков и удаление дубликатов

Проверяем наличие пропусков в целевой переменной  и признаках, а также удаляем строки-дубликаты для предотвращения утечки данных.

In [9]:
# Удаляем строки без таргета (если есть)
df_clean = df_clean.dropna(subset=["rating"])

# Удаление дубликатов
duplicates_before = df_clean.duplicated().sum()
if duplicates_before > 0:
    df_clean = df_clean.drop_duplicates()

print(f"Удалено строк-дубликатов: {duplicates_before}")
print(f"Итоговый размер обучающей матрицы: {df_clean.shape}")


Удалено строк-дубликатов: 1869
Итоговый размер обучающей матрицы: (18183, 498)


### 1.5. Экспорт списка чистых ингредиентов

Сохраняем список из 497 эталонных ингредиентов в CSV-файл для параллельного модуля работы с USDA API.

In [10]:
ingredient_list = [c for c in df_clean.columns if c != "rating"]
pd.Series(ingredient_list).to_csv("../datasets/clean_ingredients.csv", index=False, header=["ingredient"])
print(f"Список из {len(ingredient_list)} ингредиентов сохранен в datasets/clean_ingredients.csv")


Список из 497 ингредиентов сохранен в datasets/clean_ingredients.csv


## 2. Обучение моделей: Регрессия


### 2.1. Формирование обучающей и тестовой выборок, расчет Baseline

Отделяем признаки-ингредиенты от целевой оценки rating.

In [11]:
X = df_clean.drop(columns=['rating'])
y = df_clean['rating']

Делим данные на train (80%) и test (20%) со случайным сидом:

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Размер обучающей выборки (X_train): {X_train.shape}")
print(f"Размер тестовой выборки  (X_test):  {X_test.shape}")

Размер обучающей выборки (X_train): (14546, 497)
Размер тестовой выборки  (X_test):  (3637, 497)


Рассчитываем наивный baseline (прогноз среднего значения rating по обучающей выборке):

In [13]:
naive_mean = y_train.mean()

Формируем массив предсказаний наивного алгоритма для теста:

In [14]:
y_pred_naive = np.full_like(y_test, fill_value=naive_mean)

Рассчитываем и фиксируем RMSE наивного алгоритма:

In [15]:
rmse_naive = np.sqrt(mean_squared_error(y_test, y_pred_naive))
print(f"\nСредний рейтинг по базе:  {naive_mean:.4f}")
print(f"RMSE наивного алгоритма: {rmse_naive:.4f}")


Средний рейтинг по базе:  3.7157
RMSE наивного алгоритма: 1.2993


### 2.2. Сравнение базовых моделей регрессии

**Цель:** Оценить базовые алгоритмы регрессии на 5-фолдовой кросс-валидации по метрике RMSE и сравнить их с наивным baseline ().

1. **Linear Regression** (линейная регрессия без регуляризации)
2. **Ridge Regression** (линейная регрессия с L2-регуляризацией)
3. **Decision Tree Regressor** (решающее дерево)

In [16]:
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0),
    'Decision Tree': DecisionTreeRegressor(max_depth=10, random_state=42)
}

In [17]:
print(f"Базовый ориентир (Наивный RMSE): {rmse_naive:.4f}\n")
print("Результаты на кросс-валидации (5 фолдов):")

cv_scores = {}
for name, model in models.items():
    scores = cross_val_score(
        model, X_train, y_train, 
        cv=5, 
        scoring='neg_root_mean_squared_error',
        n_jobs=-1
    )
    rmse_cv = -scores.mean()
    cv_scores[name] = rmse_cv
    print(f"- {name}: RMSE = {rmse_cv:.4f} (+/- {scores.std():.4f})")

Базовый ориентир (Наивный RMSE): 1.2993

Результаты на кросс-валидации (5 фолдов):


- Linear Regression: RMSE = 1.2622 (+/- 0.0262)
- Ridge Regression: RMSE = 1.2561 (+/- 0.0244)
- Decision Tree: RMSE = 1.2933 (+/- 0.0281)


### 2.3. Подбор гиперпараметров для лучшей базовой модели (GridSearchCV)

Среди одиночных моделей лидером стала **Ridge Regression** ().
Оптимизируем коэффициент регуляризации  через  на 5 фолдах кросс-валидации.

In [18]:
# Задаем диапазон значений alpha для проверки
param_grid_ridge = {
    'alpha': [0.1, 1.0, 10.0, 50.0, 100.0, 200.0, 500.0]
}

In [19]:
grid_ridge = GridSearchCV(
    estimator=Ridge(),
    param_grid=param_grid_ridge,
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)

grid_ridge.fit(X_train, y_train)

best_alpha = grid_ridge.best_params_['alpha']
best_ridge_rmse = -grid_ridge.best_score_

print(f"Лучший параметр alpha: {best_alpha}")
print(f"RMSE Ridge после тюнинга: {best_ridge_rmse:.4f} (было 1.2561)")

Лучший параметр alpha: 50.0
RMSE Ridge после тюнинга: 1.2428 (было 1.2561)


### 2.4. Исследование ансамблей моделей (Ensemble Methods)

**Цель:** Проверить, сможет ли консилиум моделей превзойти лучшую одиночную модель (Ridge,RMSE = 1.2428).

1. **Random Forest Regressor** (бэггинг)
2. **Gradient Boosting Regressor** (бустинг)
3. **AdaBoost Regressor** (адаптивный бустинг: фокусировка на сложных объектах с наибольшей ошибкой)

In [20]:
ensembles = {
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42),
    'AdaBoost': AdaBoostRegressor(n_estimators=50, learning_rate=0.1, random_state=42)
}

In [21]:
print("Результаты ансамблей на кросс-валидации (5 фолдов):")

ensemble_scores = {}
for name, model in ensembles.items():
    scores = cross_val_score(
        model, X_train, y_train, 
        cv=5, 
        scoring='neg_root_mean_squared_error',
        n_jobs=-1
    )
    rmse_cv = -scores.mean()
    ensemble_scores[name] = rmse_cv
    print(f"- {name}: RMSE = {rmse_cv:.4f} (+/- {scores.std():.4f})")

Результаты ансамблей на кросс-валидации (5 фолдов):
- Random Forest: RMSE = 1.2506 (+/- 0.0222)
- Gradient Boosting: RMSE = 1.2399 (+/- 0.0191)
- AdaBoost: RMSE = 1.3082 (+/- 0.0181)


### 2.5. Финальная оценка лучших моделей регрессии на тестовой выборке

Оцениваем на скрытой тестовой выборке только двух победителей предварительного отбора:
1. **Ridge** ()
2. **Gradient Boosting**

In [ ]:
# 1. Обучаем лучшую модель Ridge на всем train и делаем предикт на test
best_ridge = Ridge(alpha=best_alpha)
best_ridge.fit(X_train, y_train)
y_pred_ridge = best_ridge.predict(X_test)
test_rmse_ridge = np.sqrt(mean_squared_error(y_test, y_pred_ridge))

In [23]:
# 2. Обучаем лучший ансамбль Gradient Boosting на всем train и делаем предикт на test
best_gb = GradientBoostingRegressor(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42)
best_gb.fit(X_train, y_train)
y_pred_gb = best_gb.predict(X_test)
test_rmse_gb = np.sqrt(mean_squared_error(y_test, y_pred_gb))

In [24]:
# 3. Сводная таблица результатов
results_df = pd.DataFrame({
    'Модель': ['Наивный Baseline', 'Ridge (alpha=50.0)', 'Gradient Boosting'],
    'Test RMSE': [rmse_naive, test_rmse_ridge, test_rmse_gb],
    'Улучшение относительно Baseline': [
        '0.00%',
        f"{(rmse_naive - test_rmse_ridge) / rmse_naive * 100:.2f}%",
        f"{(rmse_naive - test_rmse_gb) / rmse_naive * 100:.2f}%"
    ]
})
print("ИТОГИ ЭТАПА РЕГРЕССИИ (НА ТЕСТЕ):")
display(results_df)

ИТОГИ ЭТАПА РЕГРЕССИИ (НА ТЕСТЕ):


,Модель,Test RMSE,Улучшение относительно Baseline
0,Наивный Baseline,1.299324,0.00%
1,Ridge (alpha=50.0),1.210233,6.86%
2,Gradient Boosting,1.203969,7.34%


## 3. Обучение моделей: Классификация рейтингов

Переходим от непрерывного предсказания к задаче категоризации рецептов.

### 3.1. Классификация на 6 целых классов (0, 1, 2, 3, 4, 5)

In [25]:
# 1. Округляем рейтинги до ближайшего целого
y_int = np.round(df_clean['rating']).astype(int)

# 2. Делим на train и test со стратификацией по классам
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X, y_int, test_size=0.2, random_state=42, stratify=y_int
)

# 3. Смотрим распределение классов в обучающей выборке
print("Распределение классов в train:")
print(pd.Series(y_train_c).value_counts().sort_index())

Распределение классов в train:
rating
0    1294
1     116
2     472
3    1084
4    9680
5    1900
Name: count, dtype: int64


#### 3.1.1. Наивный классификатор (Baseline по Accuracy)

Самым частым классом в обучающей выборке является оценка `4`.  
Наивный алгоритм работает по правилу мажоритарного класса: на любой рецепт из тестовой выборки он отвечает оценкой `4`.  
Рассчитаем долю правильных ответов (Accuracy) этого наивного подхода, чтобы получить планку, которую должны превзойти реальные модели.

In [26]:
# Находим самый частый класс (моду) в обучающей выборке
most_frequent_class = y_train_c.mode()[0]

# Наивный робот ставит этот класс всем рецептам из test
y_pred_naive_c = np.full_like(y_test_c, fill_value=most_frequent_class)

# Считаем точность (Accuracy)
acc_naive_c = accuracy_score(y_test_c, y_pred_naive_c)

print(f"Самый частый класс: {most_frequent_class}")
print(f"Accuracy наивного классификатора: {acc_naive_c:.4f} ({acc_naive_c * 100:.2f}%)")

Самый частый класс: 4
Accuracy наивного классификатора: 0.6654 (66.54%)


#### 3.1.2. Обучение базовых моделей классификации (6 классов)

**Цель:** Проверить, способны ли алгоритмы классификации превзойти наивный baseline (`66.54%`) на задаче с 6 классами (`0, 1, 2, 3, 4, 5`).

Сравниваем три принципиально разных классификатора:
1. **Logistic Regression:** линейный вероятностный классификатор с L2-регуляризацией.
2. **Decision Tree Classifier:** дерево решений, разделяющее пространство ингредиентов на классы по критерию Джини (Gini impurity).
3. **K-Nearest Neighbors (KNN):** классификация по голосованию ближайших соседей в пространстве ингредиентов.

*Сравнение проводим на кросс-валидации (5 фолдов) по метрике Accuracy.*

In [27]:
models_c6 = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=10, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5, n_jobs=-1)
}

In [28]:
print(f"Базовый ориентир (Наивная Accuracy): {acc_naive_c:.4f}\n")
print("Результаты моделей на кросс-валидации (5 фолдов):")

cv_c6_scores = {}
for name, model in models_c6.items():
    scores = cross_val_score(
        model, X_train_c, y_train_c, 
        cv=5, 
        scoring='accuracy',
        n_jobs=-1
    )
    mean_acc = scores.mean()
    cv_c6_scores[name] = mean_acc
    print(f"- {name}: Accuracy = {mean_acc:.4f} (+/- {scores.std():.4f})")

Базовый ориентир (Наивная Accuracy): 0.6654

Результаты моделей на кросс-валидации (5 фолдов):
- Logistic Regression: Accuracy = 0.6767 (+/- 0.0014)
- Decision Tree: Accuracy = 0.6694 (+/- 0.0036)
- KNN: Accuracy = 0.6002 (+/- 0.0082)


#### 3.1.3. Подбор гиперпараметров для лучшей модели (GridSearchCV)

Лидером среди базовых классификаторов стала **Logistic Regression** с точностью `67.67%`.  
Оптимизируем её гиперпараметр регуляризации `C` (сила штрафа) с помощью `GridSearchCV` на 5 фолдах кросс-валидации.

In [29]:
param_grid_lr = {
    'C': [0.01, 0.1, 1.0, 10.0]
}

In [30]:
grid_lr = GridSearchCV(
    estimator=LogisticRegression(max_iter=1000, random_state=42),
    param_grid=param_grid_lr,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid_lr.fit(X_train_c, y_train_c)

best_c = grid_lr.best_params_['C']
best_lr_cv_acc = grid_lr.best_score_

print(f"Лучший параметр C: {best_c}")
print(f"Accuracy после тюнинга на CV: {best_lr_cv_acc:.4f} ({best_lr_cv_acc * 100:.2f}%)")

Лучший параметр C: 0.1
Accuracy после тюнинга на CV: 0.6789 (67.89%)


#### 3.1.4. Оценка лучшей модели (6 классов) на тестовой выборке

Оцениваем победителя первого этапа — **Logistic Regression (C=0.1)** — на тестовой выборке `X_test_c` и сравниваем её Accuracy с наивным baseline (`66.54%`).

In [31]:
# Обучаем лучшую модель с C=0.1 на всем train
best_lr_c6 = LogisticRegression(C=best_c, max_iter=1000, random_state=42)
best_lr_c6.fit(X_train_c, y_train_c)

# Делаем предсказание на тесте
y_pred_lr_c6 = best_lr_c6.predict(X_test_c)
test_acc_lr_c6 = accuracy_score(y_test_c, y_pred_lr_c6)

print(f"Test Accuracy (Logistic Regression C={best_c}): {test_acc_lr_c6:.4f} ({test_acc_lr_c6 * 100:.2f}%)")

Test Accuracy (Logistic Regression C=0.1): 0.6800 (68.00%)


In [32]:
results_c6 = pd.DataFrame({
    'Модель (6 классов)': ['Наивный Baseline (класс 4)', f'Logistic Regression (C={best_c})'],
    'Test Accuracy': [acc_naive_c, test_acc_lr_c6]
})

results_c6

,Модель (6 классов),Test Accuracy
0,Наивный Baseline (класс 4),0.665384
1,Logistic Regression (C=0.1),0.679956


### 3.2. Классификация на 3 бизнес-категории ('bad', 'so-so', 'great')

**Бизнес-контекст:** Для конечного пользователя приложения важнее понимать общий вердикт блюда, а не ловить разницу между 0 и 1 или 4 и 5.

Объединяем целые оценки в 3 бизнес-класса:
- `bad`: оценки 0 и 1 (провальные блюда)
- `so-so`: оценки 2 и 3 (посредственные блюда)
- `great`: оценки 4 и 5 (отличные блюда)

In [33]:
# 1. Функция перевода чисел в 3 бизнес-категории
def map_to_3classes(rating_int):
    if rating_int in [0, 1]:
        return 'bad'
    elif rating_int in [2, 3]:
        return 'so-so'
    else:
        return 'great'

# 2. Создаем новую целевую переменную
y_3class = y_int.apply(map_to_3classes)

# 3. Смотрим распределение новых классов
print("Распределение 3 бизнес-классов:")
print(y_3class.value_counts())
print("\nВ процентах от всей базы:")
print(y_3class.value_counts(normalize=True) * 100)

Распределение 3 бизнес-классов:
rating
great    14475
so-so     1945
bad       1763
Name: count, dtype: int64

В процентах от всей базы:
rating
great    79.607326
so-so    10.696805
bad       9.695870
Name: proportion, dtype: float64


In [34]:
# Делим на train и test со стратификацией по 3 классам
X_train_3c, X_test_3c, y_train_3c, y_test_3c = train_test_split(
    X, y_3class, test_size=0.2, random_state=42, stratify=y_3class
)

# Новый наивный baseline (всегда отвечать самым частым классом)
most_frequent_3c = y_train_3c.mode()[0]
y_pred_naive_3c = np.full_like(y_test_3c, fill_value=most_frequent_3c)

acc_naive_3c = accuracy_score(y_test_3c, y_pred_naive_3c)

print(f"Самый частый класс: {most_frequent_3c}")
print(f"Accuracy наивного классификатора (3 класса): {acc_naive_3c:.4f} ({acc_naive_3c * 100:.2f}%)")

Самый частый класс: great
Accuracy наивного классификатора (3 класса): 0.7960 (79.60%)


#### 3.2.1. Обучение моделей на 3 класса (Метрика: Accuracy)

Проверим, насколько базовые алгоритмы способны обойти наивный baseline (`79.60%`) при делении на 3 класса.
Сравниваем:
1. **Logistic Regression**
2. **Decision Tree Classifier**

In [45]:
models_3c = {
    'Logistic Regression': LogisticRegression(C=0.1, max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5, n_jobs=-1)
}

print(f"Базовый ориентир (Наивная Accuracy 3 класса): {acc_naive_3c:.4f}\n")
print("Результаты на кросс-валидации (5 фолдов, метрика Accuracy):")

for name, model in models_3c.items():
    scores = cross_val_score(
        model, X_train_3c, y_train_3c, 
        cv=5, 
        scoring='accuracy',
        n_jobs=-1
    )
    print(f"- {name}: Accuracy = {scores.mean():.4f} (+/- {scores.std():.4f})")

Базовый ориентир (Наивная Accuracy 3 класса): 0.7960

Результаты на кросс-валидации (5 фолдов, метрика Accuracy):
- Logistic Regression: Accuracy = 0.8048 (+/- 0.0020)
- Decision Tree: Accuracy = 0.8060 (+/- 0.0020)
- KNN: Accuracy = 0.7700 (+/- 0.0045)


In [36]:
grid_lr_3c = GridSearchCV(
    estimator=LogisticRegression(max_iter=1000, random_state=42),
    param_grid={'C': [0.01, 0.1, 1.0, 5.0]},
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid_lr_3c.fit(X_train_3c, y_train_3c)

best_c_3c = grid_lr_3c.best_params_['C']
print(f"\nЛучший C для 3 классов: {best_c_3c}")
print(f"Accuracy на CV: {grid_lr_3c.best_score_:.4f}")


Лучший C для 3 классов: 0.1
Accuracy на CV: 0.8048


In [46]:
# Оценка лучшей модели (3 класса, Accuracy) на тестовой выборке
best_lr_3c = LogisticRegression(C=best_c_3c, max_iter=1000, random_state=42)
best_lr_3c.fit(X_train_3c, y_train_3c)
test_acc_3c = accuracy_score(y_test_3c, best_lr_3c.predict(X_test_3c))

print(f"Test Accuracy (Logistic Regression C={best_c_3c}): {test_acc_3c:.4f} (Baseline: {acc_naive_3c:.4f})")

Test Accuracy (Logistic Regression C=0.1): 0.8053 (Baseline: 0.7960)


### 3.3. Оптимизация целевой бизнес-метрики: Precision класса 'great'

**Инженерное обоснование:**
В задаче рекомендации питания цена ошибки асимметрична:
- Ошибка I рода (сказать «плохо» на хорошее блюдо): пользователь просто выберет другой рецепт.
- Ошибка II рода (сказать «GREAT» на невкусное блюдо): пользователь потратит время и деньги на несъедобную еду, что разрушит доверие к сервису.

Следовательно, целевой метрикой является **Precision класса 'great'** — процент блюд, которые действительно оказались великолепными среди всех, кому модель присвоила класс `great`.

In [47]:
# Задаем функцию расчета Precision строго для класса 'great'
def precision_great_score(y_true, y_pred):
    return precision_score(y_true, y_pred, labels=['great'], average='micro', zero_division=0)
precision_great_scorer = make_scorer(precision_great_score)
models_precision = {
    'Logistic Regression': LogisticRegression(C=0.1, max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5, n_jobs=-1)
}
print("Результаты базовых моделей по метрике Precision('great') на кросс-валидации:")
for name, model in models_precision.items():
    scores = cross_val_score(
        model, X_train_3c, y_train_3c,
        cv=5,
        scoring=precision_great_scorer,
        n_jobs=-1
    )
    print(f"- {name}: Precision('great') = {scores.mean():.4f} (+/- {scores.std():.4f})")

Результаты базовых моделей по метрике Precision('great') на кросс-валидации:
- Logistic Regression: Precision('great') = 0.8096 (+/- 0.0022)
- Decision Tree: Precision('great') = 0.8078 (+/- 0.0015)
- KNN: Precision('great') = 0.8217 (+/- 0.0027)


#### 3.3.1. Исследование ансамблей для максимизации Precision('great')

**Цель:** Применить ансамблевые методы (Random Forest и Gradient Boosting), чтобы повысить точность рекомендаций великолепных блюд (`great`) и минимизировать риск ложноположительных срабатываний.

In [48]:
ensembles_clf = {
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42),
    'AdaBoost': AdaBoostClassifier(n_estimators=50, learning_rate=0.1, random_state=42)
}

In [49]:
print("Результаты ансамблей по Precision('great') на кросс-валидации (5 фолдов):")

ensemble_c_scores = {}
for name, model in ensembles_clf.items():
    scores = cross_val_score(
        model, X_train_3c, y_train_3c,
        cv=5,
        scoring=precision_great_scorer,
        n_jobs=-1
    )
    mean_score = scores.mean()
    ensemble_c_scores[name] = mean_score
    print(f"- {name}: Precision('great') = {mean_score:.4f} (+/- {scores.std():.4f})")

Результаты ансамблей по Precision('great') на кросс-валидации (5 фолдов):


- Random Forest: Precision('great') = 0.8045 (+/- 0.0014)
- Gradient Boosting: Precision('great') = 0.8118 (+/- 0.0025)
- AdaBoost: Precision('great') = 0.7961 (+/- 0.0001)


#### 3.3.2. Подбор гиперпараметров ансамбля-победителя (GridSearchCV)

Лидером по метрике `Precision('great')` стал **Gradient Boosting** с результатом `0.8118`.  
Оптимизируем его гиперпараметры (`learning_rate` и `max_depth`) с помощью `GridSearchCV` на 5 фолдах кросс-валидации под целевую бизнес-метрику.

In [40]:
param_grid_gb = {
    'learning_rate': [0.05, 0.1],
    'max_depth': [3, 5]
}

In [41]:
grid_gb = GridSearchCV(
    estimator=GradientBoostingClassifier(n_estimators=100, random_state=42),
    param_grid=param_grid_gb,
    cv=5,
    scoring=precision_great_scorer,
    n_jobs=-1
)

grid_gb.fit(X_train_3c, y_train_3c)

best_gb_params = grid_gb.best_params_
best_gb_score = grid_gb.best_score_

print(f"Лучшие параметры Gradient Boosting: {best_gb_params}")
print(f"Precision('great') на CV после тюнинга: {best_gb_score:.4f}")

Лучшие параметры Gradient Boosting: {'learning_rate': 0.1, 'max_depth': 5}
Precision('great') на CV после тюнинга: 0.8118


In [52]:
param_grid_rf = {
    'n_estimators': [100],
    'max_depth': [10, 20, None]
}

grid_rf = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid=param_grid_rf,
    cv=5,
    scoring=precision_great_scorer,
    n_jobs=-1
)

grid_rf.fit(X_train_3c, y_train_3c)

print(f"Лучшие параметры Random Forest: {grid_rf.best_params_}")
print(f"Precision('great') на CV после тюнинга: {grid_rf.best_score_:.4f}")

Лучшие параметры Random Forest: {'max_depth': None, 'n_estimators': 100}
Precision('great') на CV после тюнинга: 0.8139


#### 3.3.3. Финальная оценка лучших классификаторов на тесте (Precision 'great')

Оцениваем победителей на тестовой выборке `X_test_3c`, замеряя снайперскую точность рекомендаций (`Precision` класса `great`), а также проверяем их работу на контрольных наборах ингредиентов:
1. **Logistic Regression** (`C=0.1`) — одиночная линейная модель.
2. **Gradient Boosting** (`lr=0.1, depth=5`) — затюненный градиентный бустинг.
3. **Random Forest** (оптимальная глубина из `GridSearchCV`) — ансамбль, способный улавливать редкие комбинации ингредиентов.

In [53]:
# 1. Наивный классификатор
prec_naive = precision_great_score(y_test_3c, y_pred_naive_3c)

# 2. Logistic Regression
best_lr_final = LogisticRegression(C=0.1, max_iter=1000, random_state=42)
best_lr_final.fit(X_train_3c, y_train_3c)
prec_lr_test = precision_great_score(y_test_3c, best_lr_final.predict(X_test_3c))

# 3. Gradient Boosting
best_gb_final = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42)
best_gb_final.fit(X_train_3c, y_train_3c)
prec_gb_test = precision_great_score(y_test_3c, best_gb_final.predict(X_test_3c))

# 4. Random Forest (с оптимальной глубиной из GridSearch)
best_rf_final = grid_rf.best_estimator_
prec_rf_test = precision_great_score(y_test_3c, best_rf_final.predict(X_test_3c))

In [54]:
results_precision_final = pd.DataFrame({
    'Модель (3 класса)': ['Наивный Baseline', 'Logistic Regression (C=0.1)', 'Gradient Boosting', 'Random Forest (tuned)'],
    "Test Precision ('great')": [prec_naive, prec_lr_test, prec_gb_test, prec_rf_test]
})

results_precision_final

,Модель (3 класса),Test Precision ('great')
0,Наивный Baseline,0.795986
1,Logistic Regression (C=0.1),0.812197
2,Gradient Boosting,0.812696
3,Random Forest (tuned),0.814794


In [55]:
test_dishes = {
    'Bad dish': ['saffron', 'gin', 'chile pepper'],
    'Great dish': ['bread', 'avocado', 'cheese', 'garlic']
}

for dish_name, ings in test_dishes.items():
    x_dish = pd.DataFrame([[1 if f in ings else 0 for f in X.columns]], columns=X.columns)
    print(f"\n{dish_name} {ings}:")
    print(f"  - Logistic Regression -> {best_lr_final.predict(x_dish)[0]}")
    print(f"  - Gradient Boosting   -> {best_gb_final.predict(x_dish)[0]}")
    print(f"  - Random Forest       -> {best_rf_final.predict(x_dish)[0]}")


Bad dish ['saffron', 'gin', 'chile pepper']:
  - Logistic Regression -> great
  - Gradient Boosting   -> great
  - Random Forest       -> bad

Great dish ['bread', 'avocado', 'cheese', 'garlic']:
  - Logistic Regression -> great
  - Gradient Boosting   -> great
  - Random Forest       -> great


## 4. Решение (Decision) и Сохранение модели

### Обоснование выбора:
Для продакшн-программы `nutritionist.py` выбирается **классификация на 3 категории** (`bad`, `so-so`, `great`) на базе ансамбля **Random Forest Classifier** (с оптимальной глубиной из `GridSearchCV`), упакованного в `Pipeline`:
1. Регрессия предсказывает непрерывный рейтинг с погрешностью ~1.2 звезды, что не дает однозначного ответа пользователю.
2. Именно глубокий **Random Forest** показал наивысший `Test Precision('great') = 0.8148` и корректно распознал как плохое сочетание (`saffron, gin, chile pepper -> bad`), так и отличное (`bread, avocado, cheese, garlic -> great`).

Обучаем финальный `Pipeline` на всей выборке и сохраняем его со списком признаков в файл для продакшн-модуля `recipes.py`.

In [56]:
# 1. Собираем финальный Pipeline с лучшим классификатором (Random Forest)
final_pipeline = Pipeline([
    ('classifier', RandomForestClassifier(
        **grid_rf.best_params_,
        random_state=42,
        n_jobs=-1
    ))
])

# 2. Обучаем Pipeline на всех 18 183 рецептах
final_pipeline.fit(X, y_3class)

# 3. Формируем словарь артефактов для модуля recipes.py
model_payload = {
    'model': final_pipeline,
    'features': list(X.columns),
    'classes': list(final_pipeline.classes_)
}

# 4. Сохраняем в файл recipe_model.joblib в папке src
joblib.dump(model_payload, 'recipe_model.joblib')

print("Финал исследования в recipes.ipynb!")
print(f"- Обучен Pipeline на {X.shape[0]} рецептах и {X.shape[1]} ингредиентах")
print(f"- Модель сохранена в файл: src/recipe_model.joblib")

Финал исследования в recipes.ipynb!
- Обучен Pipeline на 18183 рецептах и 497 ингредиентах
- Модель сохранена в файл: src/recipe_model.joblib


## 5. Пищевая ценность (Nutrition Facts)

Для каждого ингредиента из `clean_ingredients.csv` тянем данные из **USDA FoodData Central** (Foundation + SR Legacy) и считаем `%DV = значение_на_100г / суточная_норма × 100`.

Результат — `nutrition_facts.csv`, где строки — ингредиенты, столбцы — нутриенты в % от Daily Value.

In [41]:
recipes_json = recipes_data
raw = df.copy()   

NOISE_COMPOSITE = {
    "casserole/gratin", "cobbler/crumble", "condiment/spread",
    "freeze/chill", "grill/barbecue", "low/no sugar",
    "poker/game night", "rosh hashanah/yom kippur",
    "soufflé/meringue", "soup/stew", "stuffing/dressing",
    "wheat/gluten-free",
}
noise_in_raw = [c for c in NOISE_COMPOSITE if c in raw.columns]
if noise_in_raw:
    raw = raw.drop(columns=noise_in_raw)
    print(f"Убраны не-ингредиенты: {len(noise_in_raw)} столбцов")

print("raw:", raw.shape)
print("  milk:", "milk" in raw.columns, "| jam:", "jam" in raw.columns)

recipes_json = recipes_data   
clean = pd.read_csv(f"{DATA}/clean_ingredients.csv")
print("clean_ingredients.csv: shape =", clean.shape)

Убраны не-ингредиенты: 12 столбцов
raw: (20052, 668)
  milk: True | jam: True
clean_ingredients.csv: shape = (497, 1)


### 5.1. Сборка `ml_df`: только столбцы-ингредиенты

Из `raw` оставляем `rating` и колонки, чьи имена есть в `clean_ingredients.csv`. Получаем 488 ингредиентов + таргет.

In [42]:
FORBIDDEN = {
    "title", "leftovers", "california", "dominican republic",
    "protein", "low cholesterol", "pan-fry", "fruit", "stew", "braise",
    "rating", "calories", "fat", "sodium", "date", "desc",
    "directions", "ingredients", "categories",
}

if "ingredient" in clean.columns:
    ingredients_set = set(clean["ingredient"].astype(str).str.lower().str.strip())
else:
    ingredients_set = set(clean.iloc[:, 0].astype(str).str.lower().str.strip())

keep_cols = ["rating"] + [
    c for c in raw.columns
    if c.lower().strip() in ingredients_set
    and c not in FORBIDDEN
    and not c.startswith("#")
]

ml_df = raw[keep_cols].copy()
print("ml_df:", ml_df.shape)
print("milk в ml_df:", "milk" in ml_df.columns)
print("jam  в ml_df:", "jam"  in ml_df.columns)

ml_df: (20052, 489)
milk в ml_df: True
jam  в ml_df: True


### 5.2. USDA API: константы и функции

- `DAILY_VALUES` — словарь `nutrientId → (имя, единица, суточная норма по FDA)`.
- `BAD_STARTS` — чёрный список префиксов описаний, под который попадают десерты/выпечка/полуфабрикаты.
- `_usda_query`, `_pick_best`, `fetch_food_nutrients` — обёртки над FoodData Central.

In [43]:
USDA_SEARCH_URL = "https://api.nal.usda.gov/fdc/v1/foods/search"

# FDC nutrientId → (название, единица, суточная норма по FDA)

DAILY_VALUES = {
    # ---------- DRVs: Food Components ----------
    1004: ("Total Fat",          "g",    78.0),
    1258: ("Saturated Fat",      "g",    20.0),
    1253: ("Cholesterol",        "mg",  300.0),
    1005: ("Total Carbohydrate", "g",   275.0),
    1093: ("Sodium",             "mg", 2300.0),
    1079: ("Dietary Fiber",      "g",    28.0),
    1003: ("Protein",            "g",    50.0),
    # Added sugars: в USDA Foundation/SR Legacy систематически отсутствует
    # (nutrientId 1235 = Sugars, total; 2000 = Sugars, added, есть далеко не всегда).
    # Колонку заводим, но фактически будет нулевая.
    2000: ("Added Sugars",       "g",    50.0),

    # ---------- RDIs: Vitamins ----------
    1106: ("Vitamin A",          "mcg", 900.0),
    1162: ("Vitamin C",          "mg",   90.0),
    1114: ("Vitamin D",          "mcg",  20.0),
    1109: ("Vitamin E",          "mg",   15.0),
    1185: ("Vitamin K",          "mcg", 120.0),
    1165: ("Thiamin",            "mg",    1.2),
    1166: ("Riboflavin",         "mg",    1.3),
    1167: ("Niacin",             "mg",   16.0),
    1170: ("Vitamin B6",         "mg",    1.7),
    1177: ("Folate",             "mcg", 400.0),   # см. комментарий ниже
    1178: ("Vitamin B12",        "mcg",   2.4),
    1176: ("Biotin",             "mcg",  30.0),
    1175: ("Pantothenic Acid",   "mg",    5.0),

    # ---------- RDIs: Minerals ----------
    1087: ("Calcium",            "mg", 1300.0),
    1089: ("Iron",               "mg",   18.0),
    1092: ("Potassium",          "mg", 4700.0),
    1091: ("Phosphorus",         "mg", 1250.0),
    1100: ("Iodine",             "mcg", 150.0),
    1090: ("Magnesium",          "mg",  420.0),
    1095: ("Zinc",               "mg",   11.0),
    1103: ("Selenium",           "mcg",  55.0),
    1098: ("Copper",             "mg",    0.9),
    1101: ("Manganese",          "mg",    2.3),
    1097: ("Chromium",           "mcg",  35.0),
    1102: ("Molybdenum",         "mcg",  45.0),
    1088: ("Chloride",           "mg", 2300.0),
    1180: ("Choline",            "mg",  550.0),
}

BAD_STARTS = (
    # десерты/выпечка
    "cake", "crackers", "cookies", "candies", "cereal",
    "snacks", "pasta", "pie", "pizza", "soup", "sausage", "fast food",
    # жиры/масла
    "oil", "shortening", "lard", "margarine",
    # молочка-полуфабрикаты
    "butter,", "cream,", "cheese, cream", "cheese, cottage",
    "cheese food", "cheese spread", "cheese sauce", "cheese dip",
    # сдобный хлеб
    "bread, egg", "bread, stuffing", "bread, cheese", "bread, raisin",
    "bread, banana", "bread, fruit", "bread, cranberry", "bread, nut",
    "bread, garlic", "bread, olive", "bread, brioche", "bread, panettone",
    # прочее
    "beverages", "babyfood",
)

def _usda_query(q, api_key):
    try:
        r = requests.post(
            USDA_SEARCH_URL,
            params={"api_key": api_key},
            json={"query": q, "pageSize": 15,
                  "dataType": ["Foundation", "SR Legacy"]},
            timeout=30,
        )
        if r.status_code != 200:
            return []
        if "application/json" not in r.headers.get("Content-Type", ""):
            return []
        return r.json().get("foods") or []
    except (requests.RequestException, ValueError, json.JSONDecodeError):
        return []


def _pick_best(foods, q_low):
    """Лучший food. НЕ возвращает foods[0] fallback — только None если нет."""
    for f in foods:
        desc = (f.get("description") or "").lower().strip()
        if not desc or desc.startswith(BAD_STARTS):
            continue
        if desc.startswith(q_low):
            return f
    for f in foods:
        desc = (f.get("description") or "").lower().strip()
        if not desc or desc.startswith(BAD_STARTS):
            continue
        if q_low in desc.split(",")[0]:
            return f
    for f in foods:
        desc = (f.get("description") or "").lower().strip()
        if not desc or desc.startswith(BAD_STARTS):
            continue
        if q_low in desc:
            return f
    return None


def fetch_food_nutrients(query, api_key, retries=2):
    if not api_key:
        return {}
    q_low = query.lower().strip()
    candidates = [
        q_low,
        f"{q_low} raw",
        f"{q_low}s, raw",
        f"{q_low}, raw",
    ]
    for attempt in range(retries):
        for q in candidates:
            foods = _usda_query(q, api_key)
            if not foods:
                continue
            best = _pick_best(foods, q_low)
            if best is None:
                continue
            out = {}
            for n in best.get("foodNutrients", []):
                nid = n.get("nutrientId")
                val = n.get("value")
                if val is None:
                    val = n.get("amount")
                if nid is None or val is None:
                    continue
                try:
                    out[int(nid)] = float(val)
                except (TypeError, ValueError):
                    continue
            if out:
                return out
        time.sleep(1)
    return {}

### 5.3. Загрузка и достройка кэша USDA

Кэш (`usda_raw_cache.json`) экономит вызовы API: если ингредиент уже там — не запрашиваем снова. Если чего-то нет и есть ключ — дозапрашиваем.

In [44]:
CACHE_PATH = "usda_raw_cache.json"

ingredient_cols = [c for c in ml_df.columns if c != "rating"]
print("Ингредиентов в очереди:", len(ingredient_cols))

if os.path.isfile(CACHE_PATH):
    with open(CACHE_PATH, encoding="utf-8") as f:
        _raw_cache = json.load(f)
    # JSON читает ключи как строки — конвертируем в int
    nutrients_raw = {
        ing: {int(k): float(v) for k, v in (nutr or {}).items()}
        for ing, nutr in _raw_cache.items()
    }
    print(f"Кэш загружен: {len(nutrients_raw)} ингредиентов (ключи → int)")
else:
    nutrients_raw = {}
    print("Кэша нет — стартуем с нуля")

missing = [ing for ing in ingredient_cols if ing not in nutrients_raw]
if not missing:
    print("Все ингредиенты уже в кэше — API не нужен")
else:
    print(f"В кэше нет {len(missing)} ингредиентов")
    if not USDA_API_KEY:
        print("API-ключ не задан — пропускаем недостающие (для сдачи это OK)")
    else:
        for i, ing in enumerate(missing, 1):
            nutrients_raw[ing] = fetch_food_nutrients(ing, USDA_API_KEY)
            if i % 25 == 0:
                print(f"  {i}/{len(missing)}  последний: {ing}")
                with open(CACHE_PATH, "w", encoding="utf-8") as f:
                    json.dump(nutrients_raw, f, ensure_ascii=False)
                time.sleep(0.1)
        with open(CACHE_PATH, "w", encoding="utf-8") as f:
            json.dump(nutrients_raw, f, ensure_ascii=False)
        print(f"Кэш обновлён до {len(nutrients_raw)} ингредиентов")

found = sum(1 for v in nutrients_raw.values() if v)
print(f"Итого: {len(nutrients_raw)} ингредиентов, из них с нутриентами: {found}")

# Санити: проверим, что в кэше int-ключи 4-значные
sample = nutrients_raw.get("almond", {})
print("\nКонтроль almond:")
print("  [1003] protein:", sample.get(1003))
print("  [1087] calcium:", sample.get(1087))
print("  [203]  старый SR:", sample.get(203), "(должно быть None)")

Ингредиентов в очереди: 488
Кэш загружен: 496 ингредиентов (ключи → int)
Все ингредиенты уже в кэше — API не нужен
Итого: 496 ингредиентов, из них с нутриентами: 418

Контроль almond:
  [1003] protein: 26.2
  [1087] calcium: 232.0
  [203]  старый SR: None (должно быть None)


### 5.4. Сборка `nutrition_facts.csv`

Считаем `%DV = value / daily × 100` для каждого нутриента из `DAILY_VALUES`, сохраняем CSV с суффиксом `(%DV)` в названиях столбцов.

In [45]:
NUTRITION_CSV = "nutrition_facts.csv"

rows = []
for ing, nutr in nutrients_raw.items():
    if not nutr:
        continue
    row = {"ingredient": ing}
    for nid, (name, unit, daily) in DAILY_VALUES.items():
        value = nutr.get(nid, 0.0) or 0.0
        # %DV = (значение_на_100г / суточная_норма) * 100
        row[name] = round(value / daily * 100, 2) if daily else 0.0
    rows.append(row)

nutrition_df = pd.DataFrame(rows).set_index("ingredient")

# --- обязательные проверки (до переименования — по каноническим именам) ---
assert nutrition_df.shape[0] > 50, f"Мало ингредиентов: {nutrition_df.shape[0]}"
assert nutrition_df.shape[1] >= 3,  f"Мало нутриентов: {nutrition_df.shape[1]}"
assert nutrition_df["Protein"].max() > 1.0, "Все нули — проверь DAILY_VALUES/кэш"

# --- переименовываем колонки: «Protein» → «Protein (%DV)» ---
nutrition_df_pct = nutrition_df.add_suffix(" (%DV)")
nutrition_df_pct.to_csv(NUTRITION_CSV)

print(f" Сохранено: {NUTRITION_CSV}")
print(f"   shape: {nutrition_df_pct.shape}")
print(f"   колонки: {list(nutrition_df_pct.columns)[:5]} ... (всего {nutrition_df_pct.shape[1]})")

# --- печать с явным %DV (уже в заголовках) ---
pd.set_option("display.width", 160)
print("\nПервые 3 строки (% Daily Value):")
print(nutrition_df_pct.head(3))

# --- контрольные значения ---
print("\nКонтрольные значения (% DV):")
for ing in ["almond", "bread", "avocado", "cheese", "garlic"]:
    if ing in nutrition_df_pct.index:
        r = nutrition_df_pct.loc[ing]
        print(f"  {ing:10s}  "
              f"Protein={r['Protein (%DV)']:6.2f}%  "
              f"Fat={r['Total Fat (%DV)']:6.2f}%  "
              f"Carbs={r['Total Carbohydrate (%DV)']:6.2f}%  "
              f"Ca={r['Calcium (%DV)']:6.2f}%")

 Сохранено: nutrition_facts.csv
   shape: (418, 35)
   колонки: ['Total Fat (%DV)', 'Saturated Fat (%DV)', 'Cholesterol (%DV)', 'Total Carbohydrate (%DV)', 'Sodium (%DV)'] ... (всего 35)

Первые 3 строки (% Daily Value):
                      Total Fat (%DV)  Saturated Fat (%DV)  Cholesterol (%DV)  Total Carbohydrate (%DV)  Sodium (%DV)  Dietary Fiber (%DV)  Protein (%DV)  \
ingredient                                                                                                                                                  
3-ingredient recipes             0.37                 0.00                0.0                      1.57         10.26                  0.0           1.72   
30 days of groceries            38.46                59.00               26.0                      0.00          2.87                  0.0          28.80   
alaska                           1.27                 1.19               20.0                      0.02          5.00                  0.0          34.

## 6. Похожие рецепты (Similar Recipes)

Для каждого рецепта из `full_format_recipes.json`:
1. Нормализуем список ингредиентов → множество канонических имён.
2. Строим URL вида `https://www.epicurious.com/recipes/food/views/{slug}`.
3. Сохраняем в `similar_recipes.csv`.

Похожесть запроса на рецепт считается через **коэффициент Жаккара**, порог `min_score=0.4` отсекает «непохожие».

In [46]:
sample_keys = set()
for r in recipes_json[:50]:
    sample_keys.update(r.keys())
print("Ключи:", sorted(sample_keys))

id_like = [k for k in sample_keys
           if any(x in k.lower() for x in ["id", "url", "link", "slug"])]
print("Похожие на ID:", id_like)
print("Примеры:", {k: recipes_json[0].get(k) for k in id_like})

Ключи: ['calories', 'categories', 'date', 'desc', 'directions', 'fat', 'ingredients', 'protein', 'rating', 'sodium', 'title']
Похожие на ID: []
Примеры: {}


### 6.1. Жаккар и smoke-тесты

Функция `top_similar` возвращает до 3 ближайших рецептов. Два теста:
- короткий набор ингредиентов → должен найти 3 рецепта;
- длинный набор из разных категорий → пусто (Жаккар < 0.4).

In [47]:
EPI_NON_INGR = {
    "rating", "calories", "protein", "fat", "sodium",
    "title", "date", "desc", "directions", "ingredients", "categories",
    "leftovers", "california", "dominican republic", "low cholesterol",
    "pan-fry", "fruit", "stew", "braise",
}
canon_sorted = sorted(
    [c.lower() for c in raw.columns
     if c.lower() not in EPI_NON_INGR and not c.startswith("#")],
    key=len, reverse=True
)
print("canon_sorted:", len(canon_sorted),
      "| milk:", "milk" in canon_sorted,
      "| jam:",  "jam"  in canon_sorted)


def normalize_ingredients(raw_list) -> set:
    if raw_list is None:
        return set()
    if isinstance(raw_list, float) and pd.isna(raw_list):
        return set()
    parts = re.split(r"[|,]", raw_list) if isinstance(raw_list, str) else list(raw_list)
    text = " " + " ".join(str(x).lower() for x in parts) + " "
    text = re.sub(r"[^a-zа-яё\s-]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return {c for c in canon_sorted if " " + c + " " in text}


def make_url(title: str, rid=None) -> str:
    slug = re.sub(r"[^\w\s-]", "", title.lower().strip())
    slug = re.sub(r"[\s_]+", "-", slug)
    slug = re.sub(r"-+", "-", slug).strip("-")
    if rid:
        return f"https://www.epicurious.com/recipes/food/views/{slug}-{rid}"
    return f"https://www.epicurious.com/recipes/food/views/{slug}"


rows = []
for r in recipes_json:
    title  = (r.get("title") or "").strip()
    rating = r.get("rating")
    ings   = r.get("ingredients") or []
    if not title or rating is None or not ings:
        continue

    ingredients_joined = "|".join(str(i).strip().lower() for i in ings)
    norm_set = normalize_ingredients(ingredients_joined)
    if not norm_set:
        continue

    rows.append({
        "title":       title,
        "rating":      float(rating),
        "ingredients": ingredients_joined,
        "url":         make_url(title, r.get("id")),
        "norm_set":    "|".join(sorted(norm_set)),
    })

sim_df = pd.DataFrame(rows)
print("sim_df:", sim_df.shape)
print("пустых url:", sim_df["url"].isna().sum())
print("пустых norm_set:", (sim_df["norm_set"] == "").sum())

canon_sorted: 651 | milk: True | jam: True
sim_df: (19908, 5)
пустых url: 0
пустых norm_set: 0


In [48]:
def jaccard(a: set, b: set) -> float:
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)


def top_similar(query_ingredients, sim_df, top_n=3, min_score=0.4):
    q = normalize_ingredients(query_ingredients)
    if not q:
        return pd.DataFrame()

    scores = sim_df["norm_set"].apply(
        lambda s: jaccard(q, set(s.split("|")) if s else set())
    )
    best = scores.sort_values(ascending=False).head(top_n)

    if best.iloc[0] < min_score:
        return pd.DataFrame()      # «похожих рецептов нет»

    return sim_df.loc[best.index, ["title", "rating", "url"]].assign(score=best.values)


print("=== КОРОТКИЙ набор (должен найтись) ===")
print(top_similar(["bread", "avocado", "cheese", "garlic"], sim_df).to_string())

print("\n=== ДЛИННЫЙ набор (должен вернуть пусто) ===")
print(top_similar(
    ["bread", "avocado", "cheese", "garlic", "saffron", "gin",
     "chile pepper", "banana", "vodka", "wok", "yogurt", "zucchini"],
    sim_df,
).to_string())

=== КОРОТКИЙ набор (должен найтись) ===
                                      title  rating                                                                                  url  score
1938  Sauteed Mushrooms with Melted Stilton   4.375  https://www.epicurious.com/recipes/food/views/sauteed-mushrooms-with-melted-stilton    0.6
3068                  Avocado Egg-in-a-Hole   3.750                  https://www.epicurious.com/recipes/food/views/avocado-egg-in-a-hole    0.5
8113      Ripe Olive and Goat Cheese Toasts   4.375      https://www.epicurious.com/recipes/food/views/ripe-olive-and-goat-cheese-toasts    0.5

=== ДЛИННЫЙ набор (должен вернуть пусто) ===
Empty DataFrame
Columns: []
Index: []


In [49]:
assert sim_df["url"].isna().sum() == 0
assert (sim_df["url"].str.strip() != "").all()
sim_df.to_csv(f"{OUT}/similar_recipes.csv", index=False)
print("Сохранено: similar_recipes.csv, строк:", len(sim_df))

Сохранено: similar_recipes.csv, строк: 19908


## 7. Дневное меню (Bonus)

Для каждого рецепта:
- определяем meal_type по полю `categories` из JSON (Breakfast/Brunch → breakfast, Lunch → lunch, Dinner/Main Course → dinner);
- считаем %DV для 4 ключевых нутриентов по значениям `calories/fat/protein/sodium` из JSON (FDA: 2000 kcal / 78 г / 50 г / 2300 мг).

Результат — `menu_recipes.csv`.

In [50]:
FDA_DAILY = {"calories": 2000.0, "fat": 78.0, "protein": 50.0, "sodium": 2300.0}

def classify_meal(categories):
    if not categories:
        return None
    cats = {c.lower() for c in categories}
    if cats & {"breakfast", "brunch"}:
        return "breakfast"
    if "lunch" in cats:
        return "lunch"
    if cats & {"dinner", "main course", "main-course"}:
        return "dinner"
    return None


menu_rows = []
for r in recipes_json:
    title  = (r.get("title") or "").strip()
    rating = r.get("rating")
    meal   = classify_meal(r.get("categories") or [])
    if not title or rating is None or not meal:
        continue

    cal  = float(r.get("calories") or 0)
    fat  = float(r.get("fat")      or 0)
    prot = float(r.get("protein")  or 0)
    sod  = float(r.get("sodium")   or 0)
    ings = r.get("ingredients") or []
    if not ings:
        continue

    menu_rows.append({
        "title":        title,
        "rating":       float(rating),
        "url":          make_url(title, r.get("id")),
        "meal_type":    meal,
        "ingredients":  "|".join(str(i).strip().lower() for i in ings),
        "calories_pct": round(cal  / FDA_DAILY["calories"] * 100, 2),
        "fat_pct":      round(fat  / FDA_DAILY["fat"]      * 100, 2),
        "protein_pct":  round(prot / FDA_DAILY["protein"]  * 100, 2),
        "sodium_pct":   round(sod  / FDA_DAILY["sodium"]   * 100, 2),
    })

menu_df = pd.DataFrame(menu_rows)
menu_df.to_csv("menu_recipes.csv", index=False)

print("menu_df:", menu_df.shape)
print(menu_df["meal_type"].value_counts())
print(menu_df.head(3)[["title", "meal_type", "rating",
                       "calories_pct", "fat_pct", "protein_pct", "sodium_pct"]])

menu_df: (4310, 9)
meal_type
dinner       2092
lunch        1232
breakfast     986
Name: count, dtype: int64
                                               title  meal_type  rating  calories_pct  fat_pct  protein_pct  sodium_pct
0                    Mahi-Mahi in Tomato Olive Sauce     dinner   5.000           0.0     0.00          0.0        0.00
1  Ham Persillade with Mustard Potato Salad and M...      lunch   3.750          30.1    52.56         46.0       73.74
2                                  Spicy Noodle Soup  breakfast   4.375           0.0     0.00          0.0        0.00


## 8. Верификация артефактов

Проверяем, что все файлы созданы на своих местах, и что классы из `recipes.py` (`NutritionFacts`, `SimilarRecipes`) корректно их подхватывают.

In [51]:
for f in ["recipe_model.joblib", "nutrition_facts.csv", "similar_recipes.csv"]:
    print(f"{f:25s} exists: {os.path.isfile(f'{OUT}/{f}')}")

from recipes import NutritionFacts, SimilarRecipes

nf = NutritionFacts()
sr = SimilarRecipes()

print("\n=== NutritionFacts ===")
print("almond в базе:", nf.has("almond"))
print("missing для ['a','b','garlic']:", nf.missing(["a", "b", "garlic"]))

print("\n=== SimilarRecipes ===")
short = sr.find_similar(["bread", "avocado", "cheese", "garlic"])
for r in short:
    print(f"  {r['title']} (rating {r['rating']}) → {r['url']}")

recipe_model.joblib       exists: True
nutrition_facts.csv       exists: True
similar_recipes.csv       exists: True

=== NutritionFacts ===
almond в базе: True
missing для ['a','b','garlic']: ['a', 'b']

=== SimilarRecipes ===
  Sauteed Mushrooms with Melted Stilton (rating 4.375) → https://www.epicurious.com/recipes/food/views/sauteed-mushrooms-with-melted-stilton
  Avocado Egg-in-a-Hole (rating 3.75) → https://www.epicurious.com/recipes/food/views/avocado-egg-in-a-hole
  Ripe Olive and Goat Cheese Toasts (rating 4.375) → https://www.epicurious.com/recipes/food/views/ripe-olive-and-goat-cheese-toasts
